# Clinical Documentation Agent with LangChain

This notebook walks through building an AI **agent** that processes doctor–patient transcripts and generates structured clinical documentation.

## What is an agent?

A regular LLM call is a single round-trip: you send a prompt, you get a response. An **agent** is different — it has access to **tools**, and it autonomously decides which tools to call, in what order, and when it has enough information to give a final answer. This decision loop is called the **agentic loop**.

## What this agent does

Given a doctor–patient transcript, the agent:
1. **Parses** the raw conversation into structured speaker turns
2. **Generates** a SOAP note — a standard clinical documentation format (Subjective, Objective, Assessment, Plan)
3. **Maps** diagnoses to ICD-10-CM billing codes
4. **Validates** those codes against an official ~75,000-entry database
5. **Extracts** follow-up actions for the care team (medications, labs, referrals)

The agent also uses **judgment** — for non-clinical conversations (e.g., appointment scheduling), it skips the clinical tools entirely.

## Tech stack
- **LangChain** — tool binding, message formatting, and the LLM interface
- **GPT-4o** via `langchain_openai` — the underlying model
- **Pydantic** — structured output validation
- **ACI-Bench dataset** — real doctor–patient transcripts for testing

In [1]:
## Import Necessary Libraries

import sys

sys.path.append("/Users/natasha/Documents/Echoes In AI")  # Update this path as per your setup (for me my claude_key.py is in this folder)

import os
import csv
import anthropic
import pandas as pd
import json
import re
import time
from datetime import datetime
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.tools import tool
from IPython.display import display, Markdown
from langchain_openai import ChatOpenAI
from chatgpt_key import OPENAI_API_KEY

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

import warnings
warnings.filterwarnings('ignore')

## Step 1 — Initialize the LLM

We use `ChatOpenAI` from `langchain_openai` to connect to GPT-4o.

**Key setting — `temperature=0`:** This makes the model deterministic. For clinical documentation, we want consistent, reproducible output — not creative variation. A temperature of `0` means the model always picks the highest-probability token rather than sampling randomly.

Later, we'll call `.bind_tools()` on this client to register our agent tools with the model.

In [2]:
## Initialize OpenAI Chat Client

openai_chat_client = ChatOpenAI(
    model="gpt-4o",
    api_key = OPENAI_API_KEY,
    temperature=0,
)

In [3]:
# ## Testing the Chat Client 

# response = openai_chat_client.invoke("Tell me a fun fact about AI.")
# print("====== Response: =======")
# display(Markdown(response.content))

## Step 2 — Load the Data

Two datasets are used in this project:

**ICD-10-CM codes** (`icd10cm_codes_2026.txt`): The official coding system used for medical billing in the US. We parse ~75,000 codes into a lookup dictionary: `{ "I10": "Essential (primary) hypertension", ... }`

**ACI-Bench transcripts** (`train.csv`): A benchmark dataset of real doctor–patient conversations used for clinical NLP research. Each row contains a `dialogue` (the transcript) and a `note` (the reference clinical note).

In [4]:
# Load your ICD-10 .txt file
icd10_lookup = {}
with open("Data/icd10cm_codes_2026.txt", "r", encoding="utf-8") as f:  
    for line in f:
        line = line.strip()
        if not line:
            continue
        # Split on first whitespace: code is first token, rest is description
        parts = line.split(None, 1)
        if len(parts) == 2:
            code_raw, desc = parts
            # Insert dot: A000 → A00.0, E1165 → E11.65
            if len(code_raw) > 3 and "." not in code_raw:
                code = code_raw[:3] + "." + code_raw[3:]
            else:
                code = code_raw
            icd10_lookup[code] = desc.strip()

print(f" ICD-10: {len(icd10_lookup)} codes loaded")
print(f"  Sample: A00.0 → {icd10_lookup.get('A00.0', 'NOT FOUND')}")
print(f"  Sample: E11.65 → {icd10_lookup.get('E11.65', 'NOT FOUND')}")
print(f"  Sample: I10 → {icd10_lookup.get('I10', 'NOT FOUND')}")
print()

# Load transcripts
transcripts_df = pd.read_csv("./Data/train.csv")
print("=" * 50)
print()
print(f"ACI-Bench: {len(transcripts_df)} transcripts loaded")
print(f"Columns: {list(transcripts_df.columns)}")
print(f"\nFirst transcript preview:\n{transcripts_df.iloc[0]['dialogue'][:300]}...")

 ICD-10: 74719 codes loaded
  Sample: A00.0 → Cholera due to Vibrio cholerae 01, biovar cholerae
  Sample: E11.65 → Type 2 diabetes mellitus with hyperglycemia
  Sample: I10 → Essential (primary) hypertension


ACI-Bench: 67 transcripts loaded
Columns: ['dataset', 'encounter_id', 'dialogue', 'note']

First transcript preview:
[doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?
[patient] okay .
[doctor] martha is a 50-year-old female with a past medical history significant for...


## Step 3 — Define Data Models with Pydantic

[Pydantic](https://docs.pydantic.dev/) lets us define the **shape** of our data as typed Python classes. If the LLM returns data that doesn't match the schema, we get a clear error immediately instead of silent bugs downstream.

Here's what each model represents:

| Model | What it stores |
|---|---|
| `SOAPNote` | A clinical note split into 4 standard sections |
| `ICD10Code` | A single diagnosis code with confidence and validation status |
| `FollowUpAction` | A care task: medication, lab, imaging, referral, etc. |
| `ToolCallLog` | Observability metadata for a single tool invocation |
| `AgentState` | The complete running state of the agent across all iterations |

**Tip — `Literal` types:** Fields like `confidence: Literal["high", "medium", "low"]` act as enums. Pydantic rejects any value outside the allowed set — a simple way to constrain LLM outputs to valid categories.

In [5]:
class SOAPNote(BaseModel):
    """A structured clinical note with four standard sections."""
    subjective: str
    objective: str
    assessment: str
    plan: str

class ICD10Code(BaseModel):
    """A single ICD-10-CM diagnosis code with confidence and validation metadata."""
    model_config = {"frozen": False}
    code: str
    description: str
    confidence: Literal["high", "medium", "low"] = "medium"
    is_primary: bool = False
    validated: bool = False

class FollowUpAction(BaseModel):
    """A single follow-up care task extracted from the SOAP plan."""
    type: Literal["medication", "lab", "imaging", "referral", "follow_up", "patient_education"]
    description: str
    urgency: Literal["stat", "urgent", "routine", "conditional"]

class ToolCallLog(BaseModel):
    """Observability record for a single tool invocation."""
    tool_name: str
    input_summary: str
    output_summary: str
    success: bool = True
    error: str | None = None

class AgentState(BaseModel):
    """
    Tracks the complete running state of the agent across all iterations.

    Each field is populated by the corresponding tool as the agent progresses.
    A value of `None` means that step hasn't run yet (or was skipped as unnecessary).
    """
    parsed_transcript: dict | None = None
    soap_note: SOAPNote | None = None
    icd10_codes: list[ICD10Code] | None = None
    follow_up_actions: list[FollowUpAction] | None = None
    tool_calls: list[ToolCallLog] = []
    is_complete: bool = False

    @property
    def completed_steps(self) -> list[str]:
        steps = []
        if self.parsed_transcript: steps.append("parse_transcript")
        if self.soap_note: steps.append("generate_soap")
        if self.icd10_codes: steps.append("code_diagnoses")
        if self.icd10_codes and any(c.validated for c in self.icd10_codes):
            steps.append("validate_icd10")  # ← check validated flag
        if self.follow_up_actions: steps.append("extract_actions")
        return steps

print("All models defined")

All models defined


## Step 4 — LLM Helper Utilities

Two small internal utilities used throughout the agent:

- **`_call_gpt(prompt)`** — Wraps the LangChain chat client with a single user message. Returns the response text along with token counts and latency for observability.
- **`_extract_json(text)`** — LLMs commonly wrap JSON in markdown code fences (` ```json ... ``` `). This function strips those fences so we can reliably parse the raw JSON string.

In [6]:
# ── LLM helper ────────────────────────────────────────────────────────────

def _call_gpt(prompt: str) -> str:
    """Call the LLM with a single user message and return the response text."""
    resp = openai_chat_client.invoke(
        input=[{"role": "user", "content": prompt}]
    )
    return resp.content

def _extract_json(text: str) -> str:
    """
    Strip markdown code fences from an LLM response and return the raw JSON string.

    LLMs often wrap JSON output in ```json ... ``` blocks. This function extracts
    just the content inside those fences so it can be safely passed to json.loads().
    If no fences are found, the text is returned stripped of leading/trailing whitespace.
    """
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    return m.group(1).strip() if m else text.strip()

## Step 5 — Define the Agent's Tools

This is the core of the LangChain tool system. The **`@tool` decorator** converts a regular Python function into something the LLM can discover and invoke.

### How LangChain reads your tool

When you write `@tool`, LangChain automatically extracts:
- **Function name** → the tool's identifier (`"generate_soap"`, `"validate_icd10"`, etc.)
- **Docstring** → the description the LLM uses to decide *when and whether* to call this tool
- **Type annotations** → the parameter schema the LLM uses to construct valid arguments

> **The docstring is your API contract with the LLM.** The model never sees your implementation — only the name and docstring. Write them like instructions to a careful colleague.

### Tools defined here

| Tool | When the agent calls it |
|---|---|
| `parse_transcript` | First, always — structures the raw dialogue |
| `generate_soap` | If the encounter is clinical |
| `code_diagnoses` | If the SOAP assessment contains diagnoses |
| `validate_icd10` | After coding — checks codes exist in the official database |
| `extract_actions` | If the plan contains actionable follow-up items |

In [7]:
# ── TOOLS ─────────────────────────────────────────────────────────────────

@tool
def parse_transcript(transcript: str) -> str:
    """
    Parse a raw clinical conversation into structured speaker turns.
    Call this first for any transcript.
    """

    turns = []
    for line in transcript.strip().split("\n"):
        line = line.strip()
        if not line: continue
        line = re.sub(r"^\[\d{2}:\d{2}(?::\d{2})?\]\s*", "", line)
        m = re.match(r"^\[(doctor|patient|dr\.|physician|nurse)\]\s*", line, re.I)
        if m:
            speaker = "Doctor" if m.group(1).lower() in ("doctor", "dr.", "physician") else "Patient"
            turns.append({"speaker": speaker, "text": line[m.end():].strip()})
        elif turns:
            turns[-1]["text"] += " " + line

    return json.dumps({
        "turns": turns,
        "turn_count": len(turns),
        "speaker_count": len(set(t["speaker"] for t in turns))
    })

@tool
def generate_soap(transcript: str) -> str:
    """
    Generate a SOAP note from a transcript.
    Only call if the transcript contains a clinical encounter —
    skip for administrative, scheduling, or non-medical conversations.
    """

    prompt = f"""You are a medical scribe. Generate a SOAP note from this transcript.
Return ONLY JSON: {{"subjective":"...","objective":"...","assessment":"...","plan":"...","requires_coding":true|false}}
- Set requires_coding to true if the assessment contains medical diagnoses that need ICD-10 codes
- Set requires_coding to false if the encounter is administrative or has no diagnoses

Be precise. Don't invent findings not in the transcript.

IMPORTANT: The assessment section must list diagnoses as a clean numbered list
using precise medical terminology suitable for ICD-10 coding.

Transcript:
{transcript}"""

    text = _call_gpt(prompt)
    try:
        data = json.loads(_extract_json(text))
        SOAPNote(**{k: v for k, v in data.items() if k != "requires_coding"})  # validate
        return json.dumps(data)
    except Exception as e:
        return json.dumps({"error": str(e), "raw": text[:500]})


@tool
def code_diagnoses(assessment_text: str, instructions: str = "") -> str:
    """
    Map diagnoses from a SOAP Assessment to ICD-10-CM codes.
    Only call if the SOAP note contains actual diagnoses or clinical findings —
    skip if assessment is administrative or has no medical conditions.
    """

    extra = f"\nAdditional instructions: {instructions}" if instructions else ""
    prompt = f"""You are a medical coder. Map this assessment to ICD-10-CM codes.
Use the most specific code available. Return ONLY a JSON array:
[{{"code":"X00.00","description":"...","confidence":"high|medium|low","is_primary":true|false}}]
{extra}

Assessment:
{assessment_text}"""

    text = _call_gpt(prompt)
    try:
        codes = json.loads(_extract_json(text))
        [ICD10Code(**c) for c in codes]  # validate
        return json.dumps(codes)
    except Exception as e:
        return json.dumps({"error": str(e)})


@tool
def validate_icd10(codes: list) -> str:
    """
    Validate ICD-10-CM codes against the official database.
    Only call if code_diagnoses was called and returned codes to validate.
    """

    results = []
    for code in codes:
        c = code if isinstance(code, str) else code.get("code", "")
        c = c.strip()
        if c in icd10_lookup:
            results.append({"code": c, "valid": True, "description": icd10_lookup[c]})
        else:
            parent = c.split(".")[0]
            similar = [k for k in icd10_lookup if k.startswith(parent)][:5]
            results.append({"code": c, "valid": False,
                            "note": f"Not found. Similar: {similar}" if similar else "Not found"})
    return json.dumps(results)


@tool
def extract_actions(soap_note: str) -> str:
    """
    Extract follow-up actions from a SOAP note.
    Only call if the plan contains actionable items like medications, labs,
    imaging, or referrals — skip if there are no follow-up items.
    """

    prompt = f"""Extract follow-up actions from this SOAP note. Return ONLY a JSON array:
[{{"type":"medication|lab|imaging|referral|follow_up|patient_education","description":"...","urgency":"stat|urgent|routine|conditional"}}]

SOAP Note:
{soap_note}"""

    text = _call_gpt(prompt)
    try:
        actions = json.loads(_extract_json(text))
        [FollowUpAction(**a) for a in actions]  # validate
        return json.dumps(actions)
    except Exception as e:
        return json.dumps({"error": str(e)})

## Step 6 — Bind Tools to the LLM

`bind_tools()` registers our tools with the LLM. Internally, LangChain serializes each tool's schema (name, description, input parameters) and sends it in every API request so the model knows what's available.

The return value `client_with_tools` behaves like the original client — but responses may now include `.tool_calls` populated by the model, not just `.content`.

**`parallel_tool_calls=False`:** Forces the model to call one tool at a time. This keeps the agent deterministic and state transitions predictable — each tool's output can inform the next tool's decision.

In [8]:
# ── Bind tools ────────────────────────────────────────────────────────────

ALL_TOOLS = [parse_transcript, generate_soap, code_diagnoses, validate_icd10, extract_actions]
client_with_tools = openai_chat_client.bind_tools(ALL_TOOLS, parallel_tool_calls=False)

print(f"{len(ALL_TOOLS)} tools defined: {[t.name for t in ALL_TOOLS]}")

5 tools defined: ['parse_transcript', 'generate_soap', 'code_diagnoses', 'validate_icd10', 'extract_actions']


## Step 7 — Build the Agent Loop

The **agentic loop** is the core pattern behind every LangChain agent.

```
User message
        │
        ▼
    LLM call
        │
        ▼
   Tool call?
   /        \
 Yes          No
  │            │
  ▼            ▼
Execute    Final answer 
  tool
  │
  ▼
Append result
to messages
  │
  └──────────────► LLM call (loop back)
```

### How the loop works in code

1. Build an initial `messages` list with the system prompt and the user's transcript
2. Call `client_with_tools.invoke(messages)` — the LLM returns an `AIMessage`
3. If `response.tool_calls` is non-empty: execute each tool, append results to `messages`, and loop
4. If `response.tool_calls` is empty: the LLM is done — its `.content` is the final summary

### LangChain message types used here

- **`AIMessage`** — returned by the LLM; appended directly to history so the model sees its own prior tool calls on the next iteration
- **`tool` role message** — the result of executing a tool; referenced by `tool_call_id` so the model knows which call it answers

### `AgentState`
This is our own tracking object — not a LangChain built-in. It stores every intermediate result (SOAP note, codes, actions) so we can inspect and display the full agent output after the loop finishes.

In [9]:
# ── Agent ─────────────────────────────────────────────────────────────────

SYSTEM_PROMPT = """You are a clinical documentation agent with access to 5 tools.

1. parse_transcript — always first
2. generate_soap — create the clinical note  
3. code_diagnoses — map Assessment to ICD-10 codes
4. validate_icd10 — verify codes exist in the database
5. extract_actions — pull follow-up items

Your goal is to produce accurate, complete clinical documentation — but only where relevant.
Use your judgment to decide which tools are necessary based on what the transcript contains.
Do not give medical advice, do not hallucinate details, and do not call tools unnecessarily.

After using the tools you deem necessary, provide a concise summary of your findings."""

MAX_ITERATIONS = 10

def run_agent(transcript: str) -> tuple[AgentState, str]:
    """
    Run the clinical documentation agent on a single transcript.

    This is the core agentic loop:
      1. Send the transcript to the LLM (with all tools available).
      2. If the LLM calls a tool, execute it and feed the result back into the conversation.
      3. Repeat until the LLM stops calling tools and produces a final text summary.

    The agent decides which tools to call based on the content of the transcript —
    it will skip clinical tools for non-medical conversations.

    Args:
        transcript: The raw doctor–patient (or other) conversation text.

    Returns:
        state: An AgentState with all intermediate results (SOAP note, codes, actions, logs).
        final_summary: The LLM's natural-language summary of the encounter.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Process this transcript:\n\n{transcript}"}
    ]
    state = AgentState()
    final_summary = ""

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"── Iteration {iteration} ", end="")

        response = client_with_tools.invoke(messages)

        # Add AIMessage directly to history
        messages.append(response)

        # No tool calls → done
        if not response.tool_calls:
            final_summary = response.content
            state.is_complete = True
            print()
            break

        # Process tool calls
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_input = tool_call["args"]
            tool_call_id = tool_call["id"]

            print(f"→ {tool_name} ", end="")

            # Find and invoke the matching tool
            tool_fn = next(t for t in ALL_TOOLS if t.name == tool_name)
            result_json = tool_fn.invoke(tool_input)

            # Update state based on which tool ran
            if tool_name == "parse_transcript":
                state.parsed_transcript = json.loads(result_json)

            elif tool_name == "generate_soap":
                data = json.loads(result_json)
                if "error" not in data:
                    # Strip requires_coding before creating SOAPNote
                    soap_data = {k: v for k, v in data.items() if k != "requires_coding"}
                    state.soap_note = SOAPNote(**soap_data)

            elif tool_name == "code_diagnoses":
                data = json.loads(result_json)
                if isinstance(data, list):
                    state.icd10_codes = [ICD10Code(**c) for c in data]

            elif tool_name == "validate_icd10":
                data = json.loads(result_json)
                if state.icd10_codes:
                    valid_set = {r["code"] for r in data if r.get("valid")}
                    for icd in state.icd10_codes:
                        if icd.code in valid_set:
                            icd.validated = True  # works because frozen=False

            elif tool_name == "extract_actions":
                data = json.loads(result_json)
                if isinstance(data, list):
                    state.follow_up_actions = [FollowUpAction(**a) for a in data]

            # Log tool call
            state.tool_calls.append(ToolCallLog(
                tool_name=tool_name,
                input_summary=str(tool_input)[:100],
                output_summary=result_json[:100],
            ))

            # Append tool result to messages so the LLM sees it on the next iteration
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call_id,
                "content": result_json,
            })

        print()

    print(f"Done: {len(state.tool_calls)} tool calls")
    return state, final_summary

print("run_agent() ready")

run_agent() ready


## Step 8 — Run the Agent on a Clinical Transcript

Now let's put it all together. We'll run the agent on the first transcript from ACI-Bench — a real clinical encounter involving a 50-year-old patient with heart failure, hypertension, and depression.

Watch the iteration log to see the agent's decision-making in real time: which tools it calls, in what order, and when it decides it has enough information to stop.

In [10]:
# Pick a transcript from the dataset
transcript = transcripts_df.iloc[0]["dialogue"]
print(f"Transcript length: {len(transcript)} chars")
print(f"Preview:\n{transcript[:500]}\n{'─'*60}\n")

# Run the agent
state, summary = run_agent(transcript)

Transcript length: 7708 chars
Preview:
[doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?
[patient] okay .
[doctor] martha is a 50-year-old female with a past medical history significant for congestive heart failure , depression and hypertension who presents for her annual exam . so , martha , it's been a year since i've seen you . how are you doing ?
[patient] i'm doing well . i've been
────────────────────────────────────────────────────────────

── Iteration 1 → parse_transcript 
── Iteration 2 → generate_soap 
── Iteration 3 → code_diagnoses 
── Iteration 4 → validate_icd10 
── Iteration 5 → extract_actions 
── Iteration 6 
Done: 5 tool calls


In [11]:
# ── Agent Trace ────────────────────────────────────────────────────────────
print("=" * 60)
print("AGENT TRACE")
print("=" * 60)
for i, log in enumerate(state.tool_calls, 1):
    status = "✓" if log.success else "✗"
    print(f"  {i}. {status} {log.tool_name:<20} | {log.output_summary}")

# ── Summary ────────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("AGENT SUMMARY")
print("=" * 60)
print(summary)

# ── SOAP Note ──────────────────────────────────────────────────────────────
if state.soap_note:
    print(f"\n{'=' * 60}")
    print("SOAP NOTE")
    print("=" * 60)
    for section in ["subjective", "objective", "assessment", "plan"]:
        print(f"\n[{section.upper()}]")
        print(getattr(state.soap_note, section))

# ── ICD-10 Codes ──────────────────────────────────────────────────────────
if state.icd10_codes:
    print(f"\n{'=' * 60}")
    print("ICD-10 CODES")
    print("=" * 60)
    for c in state.icd10_codes:
        v = "✓ validated" if c.validated else "⚠ not validated"
        p = " ★ PRIMARY" if c.is_primary else ""
        print(f"  {c.code:<10} {c.description:<50} ({c.confidence}) {v}{p}")

# ── Follow-up Actions ────────────────────────────────────────────────────
if state.follow_up_actions:
    print(f"\n{'=' * 60}")
    print("FOLLOW-UP ACTIONS")
    print("=" * 60)
    for a in state.follow_up_actions:
        print(f"  [{a.urgency.upper():<12}] {a.type:<20} {a.description}")

AGENT TRACE
  1. ✓ parse_transcript     | {"turns": [{"speaker": "Doctor", "text": "hi , martha . how are you ?"}, {"speaker": "Patient", "tex
  2. ✓ generate_soap        | {"subjective": "Martha is a 50-year-old female with a history of congestive heart failure, depressio
  3. ✓ code_diagnoses       | [{"code": "I50.32", "description": "Chronic systolic (congestive) heart failure", "confidence": "hig
  4. ✓ validate_icd10       | [{"code": "I50.32", "valid": true, "description": "Chronic diastolic (congestive) heart failure"}, {
  5. ✓ extract_actions      | [{"type": "medication", "description": "Increase lisinopril to 40 mg daily", "urgency": "routine"}, 

AGENT SUMMARY
The clinical encounter with Martha, a 50-year-old female with a history of congestive heart failure, depression, and hypertension, was documented. She is doing well overall, staying active, and adhering to a low sodium diet. However, she occasionally forgets her blood pressure medication, especially during stressful 

## Step 9 — Testing Agent Judgment: Administrative Transcript

A good agent doesn't blindly run every tool. Let's give it a non-clinical conversation — a simple appointment rescheduling — and verify that it stops after parsing rather than trying to generate SOAP notes or ICD codes for a scheduling exchange.

This tests whether the agent correctly reads the tool docstrings and uses discretion about which tools are necessary.

In [12]:
transcript_admin = """
Receptionist: Good morning, how can I help you?
Patient: Hi, I need to reschedule my appointment.
Receptionist: Sure, how does Tuesday at 2pm work?
Patient: Perfect, see you then.
Receptionist: Great, you're all set. See you Tuesday.
"""

print(f"Transcript Admin, {len(transcript_admin)} chars\n")
state_admin, summary_admin = run_agent(transcript_admin)
print(f"\n{summary_admin}")

Transcript Admin, 235 chars

── Iteration 1 
Done: 0 tool calls

This transcript is an administrative conversation focused on rescheduling an appointment. No clinical information or follow-up actions are present, so no further processing is necessary.


In [13]:
# ── Agent Trace ────────────────────────────────────────────────────────────
print("=" * 60)
print("AGENT TRACE")
print("=" * 60)
for i, log in enumerate(state_admin.tool_calls, 1):
    status = "✓" if log.success else "✗"
    print(f"  {i}. {status} {log.tool_name:<20} | {log.output_summary}")

# ── Summary ────────────────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("AGENT SUMMARY")
print("=" * 60)
print(summary_admin)

# ── SOAP Note ──────────────────────────────────────────────────────────────
if state_admin.soap_note:
    print(f"\n{'=' * 60}")
    print("SOAP NOTE")
    print("=" * 60)
    for section in ["subjective", "objective", "assessment", "plan"]:
        print(f"\n[{section.upper()}]")
        print(getattr(state_admin.soap_note, section))

# ── ICD-10 Codes ──────────────────────────────────────────────────────────
if state_admin.icd10_codes:
    print(f"\n{'=' * 60}")
    print("ICD-10 CODES")
    print("=" * 60)
    for c in state_admin.icd10_codes:
        v = "✓ validated" if c.validated else "⚠ not validated"
        p = " ★ PRIMARY" if c.is_primary else ""
        print(f"  {c.code:<10} {c.description:<50} ({c.confidence}) {v}{p}")

# ── Follow-up Actions ────────────────────────────────────────────────────
if state_admin.follow_up_actions:
    print(f"\n{'=' * 60}")
    print("FOLLOW-UP ACTIONS")
    print("=" * 60)
    for a in state_admin.follow_up_actions:
        print(f"  [{a.urgency.upper():<12}] {a.type:<20} {a.description}")

AGENT TRACE

AGENT SUMMARY
This transcript is an administrative conversation focused on rescheduling an appointment. No clinical information or follow-up actions are present, so no further processing is necessary.
